# Final frozen prompt comparison on Colab

Select an A100, T4 or L4 GPU runtime, then run setup, review configuration, and explicitly run the final cell. SPLITS controls dev/test selection. The default runs v1, v2, v3 on dev, then v1, v2, v3 on test (450 classifications plus six synthetic warmups). Use the same GPU for all six runs. Rerunning creates a new batch; there is no silent resume.

These compare historical SYSTEM text under the current pipeline, not historical deterministic post-processing. Freeze prompts/policy before test; the test comparison is authorized here but must not be used for further tuning. Original baseline output-field ordering differs from today's schema; any invalid outputs remain errors. GPU allocation is recorded; inspect Ollama placement to confirm GPU offload. Do not pool latencies from A100, T4 and L4.


In [ ]:
SPLITS = ("dev", "test")  # Use ("dev",) or ("test",) to select one split.
POLICY_FILE = "policy_v1_1.md"  # Same frozen policy for all three prompts.
CONTEXT_SIZE = 6
MODEL = "qwen3:4b"
EXPERIMENT_PREFIX = "final"
WARMUP = True  # One synthetic event per run, excluded from metrics and token totals.
OUTPUT_ROOT = "/content/drive/MyDrive/policy_classifier_results"


## Set up code, model and result storage

Use a fresh GPU runtime. Private repositories require authentication before cloning. Setup downloads model weights but does not classify cases.


In [ ]:
from pathlib import Path
from google.colab import userdata, drive
import importlib, json, os, shutil, subprocess, sys, time, tempfile

ROOT = Path('/content/policy_classifier')
REPO_URL = 'https://github.com/SondosElMoataz/policy-event-classifier.git'

# 1. Clone or update private GitHub repository
token = userdata.get('GITHUB_TOKEN')
assert token, "Add GITHUB_TOKEN to Colab Secrets and enable notebook access."

with tempfile.TemporaryDirectory() as tmp:
    askpass = Path(tmp) / 'askpass.sh'
    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) echo "x-access-token" ;;\n'
        '  *) printf "%s\\n" "$GITHUB_TOKEN_FOR_GIT" ;;\n'
        'esac\n'
    )
    askpass.chmod(0o700)

    env = os.environ.copy()
    env['GIT_ASKPASS'] = str(askpass)
    env['GITHUB_TOKEN_FOR_GIT'] = token
    env['GIT_TERMINAL_PROMPT'] = '0'

    if (ROOT / '.git').exists():
        subprocess.run(
            ['git', '-C', str(ROOT), 'pull', '--ff-only'],
            env=env, check=True
        )
    else:
        if ROOT.exists():
            shutil.rmtree(ROOT)  # Remove incomplete clone only

        subprocess.run(
            ['git', 'clone', REPO_URL, str(ROOT)],
            env=env, check=True
        )

token = None
env.pop('GITHUB_TOKEN_FOR_GIT', None)

# 2. Set Python source path
os.chdir(ROOT)
sys.path = [
    p for p in sys.path
    if p not in (
        '/content/policy_classifier',
        '/content/policy_classifier_source'
    )
]
sys.path.insert(0, str(ROOT))

# 3. Install Python dependencies
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'httpx'],
    check=True
)
import httpx

# 4. Install Ollama if needed
if not shutil.which('ollama'):
    if not shutil.which('zstd'):
        subprocess.run(['apt-get', 'update', '-qq'], check=True)
        subprocess.run(['apt-get', 'install', '-y', '-qq', 'zstd'], check=True)

    subprocess.run(
        'curl -fsSL https://ollama.com/install.sh | sh',
        shell=True,
        check=True
    )

# 5. Start Ollama if not already running
OLLAMA_URL = 'http://127.0.0.1:11434'

try:
    httpx.get(f'{OLLAMA_URL}/api/version', timeout=2).raise_for_status()
except httpx.HTTPError:
    log = open('/content/ollama.log', 'a')
    subprocess.Popen(
        ['ollama', 'serve'],
        stdout=log,
        stderr=subprocess.STDOUT
    )

    for _ in range(40):
        try:
            httpx.get(f'{OLLAMA_URL}/api/version', timeout=2).raise_for_status()
            break
        except httpx.HTTPError:
            time.sleep(1)
    else:
        raise RuntimeError('Ollama did not start; inspect /content/ollama.log')

# 6. Pull Qwen3 4B
subprocess.run(['ollama', 'pull', MODEL], check=True)

# 7. Mount Google Drive for experiment results
drive.mount('/content/drive')
os.environ['MOCK'] = '0'
os.environ['OLLAMA_BASE_URL'] = OLLAMA_URL

# 8. Clear stale imports from previous experiments
for name in list(sys.modules):
    if name in ('classifier', 'eval') or name.startswith(('classifier.', 'eval.')):
        sys.modules.pop(name, None)

importlib.invalidate_caches()

# 9. Load shared evaluator and classifier
from eval.run_eval import evaluate
from eval.final_runs import run_final_suite, runtime_metadata
from classifier import core

print('\nSetup complete.')
print('Classifier:', core.__file__)
print('Evaluator:', sys.modules['eval.run_eval'].__file__)
print('Model:', MODEL)
HARDWARE = runtime_metadata()
if not HARDWARE['nvidia_smi']:
    raise RuntimeError('No NVIDIA GPU detected. Select an A100, T4 or L4 Colab GPU runtime.')
print(json.dumps(HARDWARE, indent=2))

## Review configuration


In [ ]:
from eval.run_eval import load_dataset
print(json.dumps({"splits": SPLITS, "versions": ["v1", "v2", "v3"], "policy": POLICY_FILE,
                  "context_size": CONTEXT_SIZE, "model": MODEL, "warmup": WARMUP,
                  "cases_per_version": {s: len(load_dataset(split=s)) for s in SPLITS},
                  "experiment_prefix": EXPERIMENT_PREFIX, "hardware": HARDWARE}, indent=2))


## Execute the frozen final runs

The explicit cell below runs the selected splits sequentially and saves all artifacts to Drive. Batch and per-run metadata include prompt provenance, GPU/driver and Ollama details, warmup protocol, and unique names. Inspect `ollama_ps` in run metadata for CPU/GPU placement. Latency includes subsequent retries but excludes synthetic warmups; costs exclude warmup usage. Fatal failures stop the batch and preserve completed outputs; ordinary invalid classifications remain reported in each run.


In [ ]:
# Explicitly execute all selected final runs; no evaluation happens in setup.
from IPython.display import display, JSON

def show_result(RESULT_DIR):
    display(JSON(data=json.loads((RESULT_DIR / 'metrics.json').read_text())))
    failed_cases = [json.loads(line) for line in (RESULT_DIR / "failures.jsonl").read_text().splitlines()]
    print(f'Failed or invalid cases: {len(failed_cases)}')
    if not failed_cases:
        print('No failed cases among the evaluated events.')
    for case in failed_cases:
        print(f"\nCase {case['id']} | Gold: {case['gold']} | Predicted: {case['pred']}")
        print('Gold rule:', case.get('gold_rule'), '| Predicted rules:', case.get('rules'))
        print('Explanation:', case.get('rationale') or 'No valid explanation returned.')
        if case.get('error'):
            print('Error:', case['error'])
        # Expand to inspect the original request, event, context and traceback.
        display(JSON(data=case, expanded=False))
    print('Saved all run artifacts:', RESULT_DIR)

BATCH_DIR = run_final_suite(splits=SPLITS, policy_file=POLICY_FILE, context_size=CONTEXT_SIZE,
                            model=MODEL, experiment_prefix=EXPERIMENT_PREFIX, output_root=OUTPUT_ROOT,
                            warmup=WARMUP, on_result=show_result, hardware=HARDWARE)
print('Final batch manifest:', BATCH_DIR / 'batch_manifest.json')
